In [ ]:
%load_ext autoreload
%autoreload 2

# 1. Analyse Exploratoire des Données

__Objectif Métier :__ Analyser le profil financier des emprunteurs afin de préparer les données pour un modèle de Credit Scoring (Prédiction de la Probabilité de Défaut).

__Jeu de données :__ [Give Me Some Credit de Kaggle](https://www.kaggle.com/competitions/GiveMeSomeCredit/data)

Ici, il s'agit essentiellement d'une première prise en main des données, puis du nettoyage de celles-ci (traitement des données aberrantes, manquantes...), nous pourrons ensuite observer leur distribution avant de passer au Feature engineering.

## a. Imports et premières observations
Dans un premier temps, on importe les bibliothèques utiles et le jeu de données et on assigne nos données à des variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.core.pylabtools import figsize

description_donnees = pd.read_excel("../data/raw/Data Dictionary.xls", index_col=1)
description_donnees

On comprend que la variable cible, que l'on cherche à prédire, de notre étudie sera la variabe ```SeriousDlqin2yrs``` puisqu'on voit dans sa description "Person experienced 90 days past due delinquency or worse" ce qui correspond justement à la réponse de la question à laquelle on cherche à répondre : "Si je prête de l'argent à cette personne aujourd'hui, a-t-elle de fortes chances de cesser de me rembourser dans les 24 prochains mois ?"

In [ ]:
train_credit_data = pd.read_csv("../data/raw/cs-training.csv", index_col=0, header=0)
train_credit_data.head()

In [ ]:
test_credit_data = pd.read_csv("../data/raw/cs-test.csv", index_col=0, header=0)
test_credit_data.head() # Le fait que SeriousDlqin2yrs	soit en NaN dans la data de test confirme la conclusion précédente

## b. Traitement des Données Manquantes
La prochaine étape de notre étude va être de déterminer les variables qui comportent des données manquantes, on va ensuite discuter des hypothèses potentielles de ces manques et de comment nous allons les traiter.

In [ ]:
train_credit_data.info()

In [ ]:
compteur_nan = train_credit_data.isna().mean()
print(compteur_nan.sort_values(ascending=False)*100)


On tire plusieurs informations de notre appel à la méthode info() :
- On a un total de 150000 profils clients dans les données d'entraînement
- Parmi les variables, ```MonthlyIncome``` et ```NumberOfDependents``` sont celles qui présentent des données manquantes
    - Pour ```MonthlyIncome``` : On a 19.82% de valeurs manquantes
    - Pour ```NumberOfDependents``` : on a 2.61% de valeurs manquantes

 ## i. Variable : ```MonthlyIncome```
   - __Hypothèse :__ Ici, on peut deviner plusieurs raisons potentielles qui auraient poussé un client à laisser cette case vide :
       - Complexité : Le revenu du client est complexe et ne se résume pas en un monthly income
       - Précarité : c'est un revenu faible, le client ayant peur que cette statistique lui fasse défaut, il la cache
       - Irrégularité : son revenu mensuel est très volatile et le client ne sait pas quoi mettre
   - __Nature Statistique :__ Puisqu'on a pu identifier de nombreuses raisons potentielels d'omission volontaire, on conjecture que cette variable est de type MNAR (Missing not at random)
   - __Stratégie de traitement et validation :__ On va choisir d'utiliser une méthode d'imputation couplée à un indicateur binaire, ainsi, on peut à la fois combler le manque pour notre régression logistique et prendre en compte l'origine comportementale du trou. La distribution des revenus étant fortement étalée vers la droite, choisit la médiane comme stratégie étant donné la sensibilité de la moyenne aux valeurs extrêmes.

 ## ii. Variable : ```NumberOfDependents```
 - __Hypothèse :__ Cette variable indique le nombre de personnes à charge. On suppose qu'ici, une missing value peut être due au fait que le client suppose que laisser la case vide lors du remplissage est un 0 explicite.
 - __Nature Statistique :__ Ici aussi, on a probablement un cas de MNAR.
 - __Stratégie de traitement et validation :__ Si l'hypothèse est avérée, on va faire le choix de remplacer les NaN par des 0. Pour valider l'hypothèse, on va essayer de voir si $$\mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=\text{NaN}) \approx \mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=0)$$


### Validation de l'hypothèse pour ```NumberOfDependents```
On pose $P_1 = \mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=\text{NaN})$ et $P_2 =\mathbb{P}(\text{SeriousDlqin2yrs}=1 \mid \text{NumberOfDependents}=0)$

In [ ]:
masque = train_credit_data["NumberOfDependents"].isna()
train_data_nod_na = train_credit_data.loc[masque]

p_1 = train_data_nod_na["SeriousDlqin2yrs"].mean()

train_data_nod_0 = train_credit_data.loc[train_credit_data["NumberOfDependents"] == 0]
p_2 = train_data_nod_0["SeriousDlqin2yrs"].mean()

print(f'P_1 = {p_1}\nP_2 = {p_2}\n|P_1 - P_2| = {abs(p_1-p_2)*100}')


On trouve au final que $| P_1 - P_2 | \approx 1.3 \%$, on a donc un écart assez faible pour valider notre conjecture initiale pour cette variable, on va donc appliquer la stratégie de traitement qui va être de remplacer les NaN par des 0.

In [ ]:
train_credit_data["NumberOfDependents"] = train_credit_data["NumberOfDependents"].fillna(0)
train_credit_data.info()

### Mise en place de la stratégie d'imputation pour ```MonthlyIncome```
On fait le choix d'utiliser l'outil ```SimpleImputer``` de scikit learn. On fixe ```strategy = 'median'``` et ```add_indicator = True```.

In [ ]:
from sklearn.impute import SimpleImputer

imputer_income = SimpleImputer(strategy="median",add_indicator=True)
new_income = imputer_income.fit_transform(train_credit_data[["MonthlyIncome"]])

train_credit_data["MonthlyIncome"] = new_income[:,0]
train_credit_data["MissingIncomeFlag"] = new_income[:,1]

train_credit_data.info() #Plus aucune missing value
train_credit_data["MonthlyIncome"].describe()

On conclut qu'il n'y a plus aucune valeur manquante dans notre dataset, on peut donc passer au traitement des valeurs extrêmes.

## c. Traitement des données manquantes
Ici, le but va être de repérer les valeurs aberrantes dans le jeu de données puis d'en analyser les causes possibles. Selon les conclusions, nous choisirons une méthode adaptée pour les lisser

In [ ]:
train_credit_data.describe()

In [ ]:
cols_anomalie = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "age"
]

for col in cols_anomalie:
    print(train_credit_data[col].value_counts().sort_index())

L'appel à la méthode .describe() montre trois anomalies dans la distribution de nos variables et qui ont donc besoin d'un traitement spécifique :

1. Variable `age` :
    - __Observation :__ L'âge minimum relevé dans la base est de 0 an. Après un unique count on remarque que c'est un unique cas isolé et que le reste des âges commence à 21, sûrement l'âge légal pour contracter un crédit aux Etats Unis
    - __Analyse :__ C'est évidemment impossible autant biologiquement que légalement. C'est sûrement une erreur de frappe ou d'une valeur de remplacement par défaut d'un ancien système (pour éviter les NaN par ex).
    - __Décision :__ On va remplacer la ligne par la médiane pour ne pas en perdre l'information.


2. Variable ``MonthlyIncome``
    - __Observation :__ La valeur maximale atteint 3 000 000, ce qui est totalement déconnecté du 3ème quartile (75 %) qui reste dans des chiffres raisonnables (7400 environs).
    - __Analyse :__ Cette valeur peut correspondre soit à un profil très riche (risque très faible), soit à une erreur de remplissage où le client a mis son revenu ou son patrimoine annuel au lieu de son revenu mensuel.
    - __Décision :__ On va faire le choix de plafonner cette variable pour éviter de biaiser notre régression.


3. Variables de retards de paiement (```NumberOfTime...```)
    - __Observation :__ Les variables mesurant le nombre de fois où un client a été en retard (30-59Days, 60-89Days, 90DaysLate) ont des maximums aberrants systématiquement plafonnés à 98, très éloignés du 75ème centile. Après un affichage des unique counts de chacune de ces variables, on observe que les deux valeurs aberrantes qui reviennent systématiquement sont 98 et 96.
    - __Analyse :__ Mathématiquement et financièrement, accumuler 98 fois un retard de 90 jours est impossible (la banque aurait saisi les biens bien avant). Après recherche, en particulier dans les forums de Kaggle (notamment dans [cette discussion](https://www.kaggle.com/competitions/GiveMeSomeCredit/discussion/867)), de nombreux utilisateurs ont relevé ces valeurs. Le consensus est qu'il s'agit de codes internes aux sources de ces données, ce ne sont donc pas des valeurs numériques à prendre dans leur sens littéral mais plutôt des valeurs qui ont des "sens cachés". Une observation intéressante est que le nombre de 98 est systématiquement identique pour chacun des intervalles de retard (exactement 264), pareil pour 96 (exactement 5).
    - __Décision :__ Une des méthodes les plus précises serait de faire un modèle dédié au profiling des personnes qui ont ce 98 (ou 96), mais c'est trop complexe pour cette étape. On va donc évaluer leur proportion dans le dataset pour déterminer la méthode la plus adaptée (les exclure ou les écraser).

### Traîtement de l'âge :

C'est une valeur unique, on va simplement la remplacer par la médiane des âges.

In [ ]:
age_median = train_credit_data["age"].median()
train_credit_data.loc[train_credit_data["age"]== 0, "age"] = age_median
train_credit_data["age"].value_counts().sort_index()

### Traitement des extrêmes en `MonthlyIncome`

Pour éviter que les valeurs extrêmes de cette variable ne biaisent trop nos modèles, en particulier la régression logistique, on fait le choix de les capper à un centile inférieur, que l'on va déterminer en cherchant le point d'inflexion de l'écart entre les quantiles supérieurs.

In [ ]:
liste_quantiles = [0.90,0.91,0.92,0.93,0.94,0.95,0.96,0.97,0.98,0.990, 0.995, 0.999, 1]
ecarts = pd.DataFrame({'quantile' : train_credit_data["MonthlyIncome"].quantile(liste_quantiles), 'diff' :                       train_credit_data["MonthlyIncome"].quantile(liste_quantiles).diff()})
ecarts

On observe que le point d'inflexion se situe entre le quantile 0.999 et le maximum absolu. On fait donc le choix de capper les valeurs extrêmes à la valeur du quantile 0.999 qui est de ``72760`` environs, ce qui reste une valeur plausible contrairement à l'extrême qui donne un saut de presque 3 millions. Un tel salaire mensuel parait peu probable, on peut imaginer que c'est une erreur de saisie (salaire annuel, patrimoine total...).

In [ ]:
cap = train_credit_data["MonthlyIncome"].quantile(0.999)
train_credit_data["MonthlyIncome"]=train_credit_data["MonthlyIncome"].clip(lower=0, upper=cap)
train_credit_data["MonthlyIncome"].describe()


Avec cette modification, on observe que l'écart type pour cette variable passe de 12890 à 4958, on voit donc bien que cette minorité de valeurs extrêmes avait une influence importante sur la dispersion des données, ce qui aurait pu fortement biaiser toute régression.

### Traîtement des variables de retard de paiement

Précédemment, on a vu que pour les trois variables `NumberOfTime30-59DaysPastDueNotWorse, NumberOfTime60-89DaysPastDueNotWorse` et `NumberOfTimes90DaysLate`, il y avait exactement 264 clients à 98 et 5 à 96. On cherche d'abord à savoir si ce sont les mêmes personnes ou si l'intersection entre ces groupes est vide.

In [ ]:
retard = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
    ]
masque_code = train_credit_data["NumberOfTimes90DaysLate"].isin([98,96])
retards_90 = train_credit_data.loc[masque_code]
retards_90[retard].value_counts().sort_index()



On conclut que ce sont les mêmes 269 profils clients qui affichent les codes 96 et 98 pour les trois variables. Ce nombre étant marginal (environs 0.2% de l'échantillon), on fait le choix de supprimer les clients affectés.

In [ ]:
train_credit_data = train_credit_data.loc[~masque_code]
train_credit_data.shape

On a correctement supprimé les anomalies, le dataset est réduit à 149 731 lignes au lieu de 150 000

## d. Analyse bivariée conditionnelle
Ici, nous allons observer le comportement des variables explicatives en fonction de la variable cible (`SeriousDlqin2yrs`) en utilisant la bibliothèque `seaborn`.

In [ ]:
import seaborn as sns
import math
NCOLS = 3

tags_var_etude = train_credit_data.columns.difference(["SeriousDlqin2yrs", "MissingIncomeFlag"])
y_etude = train_credit_data["SeriousDlqin2yrs"]

nrows = math.ceil(len(tags_var_etude)/NCOLS)
fig, axes = plt.subplots(nrows=nrows, ncols=NCOLS, figsize=(15, 5 * nrows))
axes_flat = axes.flatten()

for i, col in enumerate(tags_var_etude):
    sns.boxplot(x=y_etude, y=col, data=train_credit_data, ax=axes_flat[i])
    axes_flat[i].set_title(f'Distribution de {col}')

for j in range(len(tags_var_etude),len(axes_flat)):
    axes_flat[j].set_axis_off()

plt.tight_layout
plt.show()

On constate de nombreuses anomalies pour la variable `RevolvingUtilizationOfUnsecuredLines` (ratios à + de 10000), on commence donc par étudier ce cas et le traiter de manière similaire au traitement fait précédemment pour la variable `MonthlyIncome`.

In [ ]:
print(train_credit_data["RevolvingUtilizationOfUnsecuredLines"].value_counts().sort_index(ascending=False))
liste_quantiles = [0.90,0.91,0.92,0.93,0.94,0.95,0.96,0.97,0.98,0.990, 0.995,0.996,0.997,0.998, 0.999, 1]

ecarts = pd.DataFrame({'quantile' : train_credit_data["RevolvingUtilizationOfUnsecuredLines"].quantile(liste_quantiles), 'diff' :                       train_credit_data["RevolvingUtilizationOfUnsecuredLines"].quantile(liste_quantiles).diff()})
ecarts


On voit très clairement que le saut se fait dès le quantile 0.999, on applique donc le clip ici.

In [ ]:
cap = train_credit_data["RevolvingUtilizationOfUnsecuredLines"].quantile(0.998)
train_credit_data["RevolvingUtilizationOfUnsecuredLines"]=train_credit_data["RevolvingUtilizationOfUnsecuredLines"].clip(lower=0, upper=cap)

sns.boxplot(x=y_etude, y="RevolvingUtilizationOfUnsecuredLines", data=train_credit_data)

On observe un résultat plus sain et cohérent avec la réalité financière avec une distinction claire entre le comportement des bons payeurs et des mauvais payeurs, avec des médianes respectivement à environs 0.15 et 0.85.